# 🧠 AI Research Foundations - Module 11: Attention & The Transformer

Welcome to Module 11! Run this notebook interactively in **Google Colab** with free GPU/CPU runtimes.

In this notebook, we:
1. Build Scaled Dot-Product Attention from raw matrix operations
2. Visualize causal attention weights and triangular masking heatmaps
3. Implement 2D Rotary Position Embeddings (RoPE)

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")

## 1. Scaled Dot-Product Attention Implementation
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

T = 6 # Sequence length
D = 16 # Dimension
torch.manual_seed(42)
Q = torch.randn(1, T, D)
K = torch.randn(1, T, D)
V = torch.randn(1, T, D)

# Causal mask: lower triangle allows attention to past and current tokens
causal_mask = torch.tril(torch.ones(T, T))
out, weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("Output Shape:", out.shape)
print("Weights Shape:", weights.shape)

## 2. Visualizing Causal Attention Weights Heatmap

In [ ]:
tokens = ["The", "robot", "learned", "to", "solve", "math"]
attn_matrix = weights[0].detach().numpy()

plt.figure(figsize=(6, 5), dpi=100)
plt.imshow(attn_matrix, cmap="viridis", vmin=0.0, vmax=1.0)
plt.colorbar(label="Attention Probability")
plt.xticks(range(T), tokens, rotation=45)
plt.yticks(range(T), tokens)
plt.xlabel("Key Positions (Attended to)")
plt.ylabel("Query Positions (Current token)")
plt.title("Causal Self-Attention Matrix (Strict Lower Triangular)", fontweight="bold")
plt.tight_layout()
plt.show()
print("Row sums (must equal 1.0):", attn_matrix.sum(axis=-1))

## 3. Rotary Position Embeddings (RoPE) Invariance Check

In [ ]:
def rotate_2d(vec, pos):
    cos_p = math.cos(pos)
    sin_p = math.sin(pos)
    return torch.tensor([vec[0] * cos_p - vec[1] * sin_p, vec[0] * sin_p + vec[1] * cos_p])

q = torch.tensor([1.0, 0.5])
k = torch.tensor([0.2, 0.8])

# Test offset delta = 4
dot_m5_n1 = torch.dot(rotate_2d(q, 5), rotate_2d(k, 1)).item()
dot_m12_n8 = torch.dot(rotate_2d(q, 12), rotate_2d(k, 8)).item()

print(f"Dot product at pos (5, 1):   {dot_m5_n1:.6f}")
print(f"Dot product at pos (12, 8):  {dot_m12_n8:.6f}")
assert math.isclose(dot_m5_n1, dot_m12_n8, rel_tol=1e-5)
print("\n✅ Module 11 notebook completed successfully!")